In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import plotly.express as px
import sys


# Smart Plotly view of divergence summary by PRP-RA affinity-level combination
import plotly.graph_objects as go
from plotly.subplots import make_subplots

repo_root = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
if str(repo_root) not in sys.path:
    sys.path.append(str(repo_root))

from scripts.run_affinity_adjudication import (
    compute_pair_stats,
    normalize_abstract_index,
    affinity_level_fixed,
 )

In [2]:

# Locate latest benchmark run
repo_root = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
bench_root = repo_root / 'data' / 'results' / 'affinity_benchmark'
run_dirs = sorted([p for p in bench_root.iterdir() if p.is_dir()])
if not run_dirs:
    raise FileNotFoundError(f'No benchmark runs found under {bench_root}')
run_dir = run_dirs[-1]
# Load RA-to-PRP mapping from processed CSV (generated by scripts/prepare_data.py)
mapping_path = repo_root / 'data' / 'processed' / 'ra_extended_grouping_ra2025.csv'
ra_grouping_df = pd.read_csv(mapping_path)

required_cols = {'RA2025', 'Grouping'}
missing_cols = required_cols - set(ra_grouping_df.columns)
if missing_cols:
    raise ValueError(f'Missing required columns in mapping file: {missing_cols}')

ra_to_prp_map = (
    ra_grouping_df[['RA2025', 'Grouping']]
    .dropna(subset=['RA2025', 'Grouping'])
    .drop_duplicates()
    .rename(columns={'RA2025': 'RA2025_ID', 'Grouping': 'PRP_Name'})
)
ra_to_prp_map['RA2025_ID'] = pd.to_numeric(ra_to_prp_map['RA2025_ID'], errors='coerce').astype('Int64')

print('Mapping file:', mapping_path)
print('Mapping rows:', len(ra_to_prp_map))
display(ra_to_prp_map.head(10))

Mapping file: /mnt/c/Users/pgallego/OneDrive - Imperial College London/PhD/Repositories/article-classifier/data/processed/ra_extended_grouping_ra2025.csv
Mapping rows: 48


,RA2025_ID,PRP_Name
0,1,IBR
1,2,IBR
2,3,IBR
3,4,IBR
4,5,IBR
5,6,IBR
6,7,Stability
7,8,Stability
8,9,Stability
9,10,Stability


# Reading....

In [3]:
run_dir_names = ["20260422_212121_1", "20260422_212121_2", "20260422_212121_3"]  # add more run folder names here
# run_dir_names = ["20260422_212121_1"] 

repo_root = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
bench_root = repo_root / "data" / "results" / "affinity_benchmark"
run_dirs = sorted([p for p in bench_root.iterdir() if p.is_dir()])
if not run_dirs:
    raise FileNotFoundError(f"No benchmark runs found under {bench_root}")

if isinstance(run_dir_names, (str, Path)):
    run_dir_names = [run_dir_names]

selected_run_dirs = [p for p in run_dirs if p.name in {str(x) for x in run_dir_names}]
if not selected_run_dirs:
    raise FileNotFoundError(f"No matching run directories found for: {run_dir_names}")

frames = []
for run_dir in selected_run_dirs:
    ra_path_adjudication = run_dir / "adjudicated_ra_affinities.csv"
    if not ra_path_adjudication.exists():
        raise FileNotFoundError(f"Missing file: {ra_path_adjudication}")
    df = pd.read_csv(ra_path_adjudication)
    df["Run_ID"] = run_dir.name
    frames.append(df)

ra_adjudication_df = pd.concat(frames, ignore_index=True)

print("Run directories:", [p.name for p in selected_run_dirs])
print("Rows:", len(ra_adjudication_df))
display(ra_adjudication_df.head(3))

Run directories: ['20260422_212121_1', '20260422_212121_2', '20260422_212121_3']
Rows: 19198


,Abstract_Index,Document Title,RA2025_ID,RA_Question,N_Models,Affinity_Mean,Affinity_Std,Affinity_Min,Affinity_Max,Affinity_Range,agreement_mean,agreement_std,agreement_min,agreement_max,Final_Affinity,Final_Reason,Final_Method,Model_Slug,Cosine_x100_Mean,Run_ID
0,1,Scenario Reduction Network Based on Wasserstei...,1,what is the future of active power balance as ...,6,10.833333,13.570802,0.0,30.0,30.0,1.0,0.0,1.0,1.0,10.833333,NaN,ensemble_mean,NaN,18.0619,20260422_212121_1
1,1,Scenario Reduction Network Based on Wasserstei...,2,how will protection systems need to change to ...,6,7.500000,9.874209,0.0,20.0,20.0,1.0,0.0,1.0,1.0,7.500000,NaN,ensemble_mean,NaN,18.1351,20260422_212121_1
2,1,Scenario Reduction Network Based on Wasserstei...,3,how could the ibr control paradigm change from...,6,10.000000,12.649111,0.0,30.0,30.0,1.0,0.0,1.0,1.0,10.000000,NaN,ensemble_mean,NaN,5.5768,20260422_212121_1


## RA

In [4]:
# Map RA questions to PRP names using data/processed/ra_grouping_ra2025.csv, then collapse abstract/PRP pairs
ra_adjudication_aggregated_df = ra_adjudication_df.copy()
ra_adjudication_aggregated_df['RA2025_ID'] = pd.to_numeric(ra_adjudication_aggregated_df['RA2025_ID'], errors='coerce').astype('Int64')
ra_adjudication_aggregated_df['Final_Affinity'] = pd.to_numeric(ra_adjudication_aggregated_df['Final_Affinity'], errors='coerce')
ra_adjudication_aggregated_df['agreement_max'] = pd.to_numeric(ra_adjudication_aggregated_df['agreement_max'], errors='coerce')

ra_adjudication_aggregated_df = ra_adjudication_aggregated_df.merge(
    ra_to_prp_map,
    on='RA2025_ID',
    how='left'
 )

ra_adjudication_aggregated_df = ra_adjudication_aggregated_df.dropna(subset=['Abstract_Index', 'PRP_Name', 'Final_Affinity']).copy()

# Keep only rows that match the max affinity inside each (Abstract_Index, PRP_Name) group
max_affinity_per_group = ra_adjudication_aggregated_df.groupby(
    ['Abstract_Index', 'PRP_Name']
 )['Final_Affinity'].transform('max')

max_rows = ra_adjudication_aggregated_df[
    ra_adjudication_aggregated_df['Final_Affinity'].eq(max_affinity_per_group)
].copy()

ra_adjudication_aggregated_df = (
    max_rows.groupby(['Abstract_Index', 'PRP_Name'], as_index=False)
    .agg(
        Final_Affinity=('Final_Affinity', 'max'),
        RA2025_ALL_ID=('RA2025_ID', lambda s: ','.join(map(str, sorted(pd.unique(s.dropna().astype(int)))))),
        agreement_max=('agreement_max', 'max')
    )
)
ra_adjudication_aggregated_df['RA2025_ID'] = ra_adjudication_aggregated_df['RA2025_ALL_ID'].apply(lambda s: s.split(',')[0] if pd.notna(s) and s else pd.NA )

ra_adjudication_aggregated_df['Affinity_Level'] = ra_adjudication_aggregated_df['Final_Affinity'].apply(affinity_level_fixed)
display(ra_adjudication_aggregated_df)

,Abstract_Index,PRP_Name,Final_Affinity,RA2025_ALL_ID,agreement_max,RA2025_ID,Affinity_Level
0,1,CROF,58.000000,27,0.666667,27,Moderate
1,1,DER Bulk Services,9.166667,18,1.000000,18,Low
2,1,DER C&C,21.000000,12,1.000000,12,Low
3,1,IBR,10.833333,1,1.000000,1,Low
4,1,Planning,87.333333,41,1.000000,41,High
...,...,...,...,...,...,...,...
2095,300,DER C&C,30.000000,15,1.000000,15,Low
2096,300,IBR,85.000000,4,1.000000,4,High
2097,300,Planning,40.000000,46,1.000000,46,Low
2098,300,Stability,62.000000,9,0.500000,9,Moderate


In [5]:
count_df = ra_adjudication_aggregated_df.groupby(['PRP_Name', 'Affinity_Level']).size().reset_index(name='n_abstracts')

fig = px.bar(count_df[(count_df['Affinity_Level'] != 'Low')],
       x = 'PRP_Name',
       y = 'n_abstracts',
       color = 'Affinity_Level',
       title='Count by taking the sum of questions per program',
       barmode='group',
       category_orders={
        'Affinity_Level': ['Moderate', 'High'],
        'source': ['prp', 'ra']
    },)
fig.update_layout(width=700, height=500)
fig.show()

In [6]:
# max_affinity = ra_adjudication_aggregated_df.groupby(['Abstract_Index'], as_index=False)['Final_Affinity'].max()
# max_affinity = pd.merge(max_affinity, ra_adjudication_aggregated_df, on=['Abstract_Index', 'Final_Affinity'], how='inner')
# max_affinity = max_affinity.drop_duplicates(subset=['Abstract_Index', 'Final_Affinity']) 
# there were two duplicates i.e., two levels with the same max affinity level for the same abstract, but different PRP-RA combinations. This is because we took the max affinity per (abstract, PRP) group, but then some abstracts had the same max affinity across different PRPs. We want to keep only one row per abstract with the max affinity, so we drop duplicates based on Abstract_Index and Final_Affinity. This way, if there are multiple rows with the same Abstract_Index and Final_Affinity, we keep only one of them (the first one encountered).
# display(max_affinity)
first_n = 2
id_max =  ra_adjudication_aggregated_df.groupby(['Abstract_Index'], as_index=False)['Final_Affinity'].apply(lambda s: s.sort_values(ascending=False).head(first_n).index).explode('Final_Affinity')
max_affinity = ra_adjudication_aggregated_df.iloc[id_max['Final_Affinity']]

count_df = max_affinity.groupby(['PRP_Name','Affinity_Level']).size().reset_index(name='n_abstracts')
fig = px.bar(count_df,
       x = 'PRP_Name',
       y = 'n_abstracts',
       color = 'Affinity_Level',
       title='Count by taking the question with max affinity per abstract',
       barmode='group',
       category_orders={
        'Affinity_Level': ['Moderate', 'High'],
        'source': ['prp', 'ra']
    },)
fig.update_layout(width=700, height=500)
fig.show()
display(max_affinity.groupby('Affinity_Level')['Abstract_Index'].count())

Affinity_Level
High        407
Low          23
Moderate    170
Name: Abstract_Index, dtype: int64

In [7]:
target_abstracts = 105
affinity_level = "High"
random_state = 42

high_df = max_affinity.loc[max_affinity["Affinity_Level"].eq(affinity_level), ["Abstract_Index", "PRP_Name", "Final_Affinity"]].copy()

# Count availability per PRP
available_per_prp = high_df.groupby("PRP_Name")["Abstract_Index"].nunique().sort_index()
n_prps = len(available_per_prp)

# Prefer abstracts that only appear under one PRP before selecting shared abstracts
prp_count_per_abstract = high_df.groupby("Abstract_Index")["PRP_Name"].nunique()
high_df["Is_Single_PRP_Abstract"] = high_df["Abstract_Index"].map(prp_count_per_abstract.eq(1))

# Iteratively allocate slots to PRPs trying to keep groups balanced while reaching the target
available = available_per_prp.astype(int).copy()
slots = pd.Series(0, index=available.index, dtype=int)
remaining = int(target_abstracts)

# Loop: at each iteration distribute remaining slots equally across PRPs that still have availability
while remaining > 0 and available.sum() > 0:
    active = available[available > 0]
    k = len(active)
    per = remaining // k
    if per == 0:
        per = 1
    for prp in active.index.tolist():
        if remaining <= 0:
            break
        alloc = min(per, available.loc[prp], remaining)
        slots.loc[prp] += alloc
        available.loc[prp] -= alloc
        remaining -= alloc
    # continue until either remaining==0 or no availability left

# Sample according to computed `slots` per PRP, preferring singleton abstracts first
def _sample_group(g):
    n = int(slots.loc[g.name]) if g.name in slots.index else 0
    if n <= 0:
        return g.iloc[0:0]

    singleton = g[g["Is_Single_PRP_Abstract"]].sort_values(["Final_Affinity", "Abstract_Index"], ascending=[False, True])
    if len(singleton) >= n:
        return singleton.head(n)

    shared = g[~g["Is_Single_PRP_Abstract"]].sort_values(["Final_Affinity", "Abstract_Index"], ascending=[False, True])
    remaining_n = n - len(singleton)
    if remaining_n <= 0:
        return singleton

    return pd.concat([singleton, shared.head(remaining_n)], ignore_index=False)

selected_high = (
    high_df.groupby("PRP_Name").apply(_sample_group, include_groups=False)
    .reset_index(level=0)
    .sort_values(["PRP_Name", "Abstract_Index"])
    .reset_index(drop=True)
)

selected_abstract_indices = selected_high["Abstract_Index"].drop_duplicates().tolist()

print(f"PRPs: {n_prps}")
print(f"Target abstracts: {target_abstracts}")
print('Allocated per PRP (slots):')
print(slots[slots > 0])
print(f"Total selected high-affinity abstracts: {len(selected_abstract_indices)}")
if remaining > 0:
    print(f"Warning: only {int(available_per_prp.sum())} abstracts available in total; remaining unfilled slots: {remaining}")

display(selected_high)

PRPs: 7
Target abstracts: 105
Allocated per PRP (slots):
PRP_Name
CROF                 16
DER Bulk Services    15
DER C&C              14
IBR                  15
Planning             15
Stability            15
System Services      15
dtype: int64
Total selected high-affinity abstracts: 104


,PRP_Name,Abstract_Index,Final_Affinity,Is_Single_PRP_Abstract
0,CROF,5,78.000000,True
1,CROF,15,72.000000,True
2,CROF,34,78.000000,True
3,CROF,40,86.833333,True
4,CROF,72,86.666667,True
...,...,...,...,...
100,System Services,214,82.000000,True
101,System Services,220,95.000000,False
102,System Services,281,91.833333,False
103,System Services,287,85.000000,False


In [8]:
fig = px.box(selected_high,
       x = 'PRP_Name',
       y = 'Final_Affinity',
    #    color = 'Affinity_Level',
    #    title='Count by taking the sum of questions per program',
    #    barmode='group',
       category_orders={
        'Affinity_Level': ['Moderate', 'High'],
        'source': ['prp', 'ra']
    },)
fig.update_layout(width=700, height=500)
fig.show()

In [9]:
abstracts_root = repo_root / "data" / "processed" / "abstracts_cleaned.csv"
abstracts_df = pd.read_csv(abstracts_root)
selected_abstracts_df = abstracts_df[abstracts_df["Source_Index"].isin(selected_high.Abstract_Index)][['Source_Index', 'Document Title', 'Abstract']].copy()
selected_abstracts_df['Program'] = None
persons = ['Abdelrahman', 'Andrey', 'Gabriel', 'Pablo']
# selected_abstracts_df = selected_abstracts_df.copy()
selected_abstracts_df["assigned to"] = [persons[i % len(persons)] for i in range(len(selected_abstracts_df))]
display(selected_abstracts_df)

selected_high['Source_Index'] = selected_high['Abstract_Index'].astype(int)
selected_abstracts_df = pd.merge(selected_abstracts_df, selected_high[['Source_Index','PRP_Name']], on='Source_Index', how='inner')
selected_abstracts_df.sort_values('assigned to').to_csv('selected_abstracts_batch_1.csv', index=False)

,Source_Index,Document Title,Abstract,Program,assigned to
0,1,Scenario Reduction Network Based on Wasserstei...,Power systems with high penetration of renewab...,None,Abdelrahman
1,2,Domain of Stability Characterization for Power...,"In this article, we approach the problem of st...",None,Andrey
4,5,Competitive Equilibrium Analysis for Renewable...,With the increasing interdependence among the ...,None,Gabriel
9,10,Achieving Emission-Reduction Goals: Multi-Peri...,Stochastic adaptive robust optimization is cap...,None,Pablo
14,15,Look-Ahead Rolling Economic Dispatch Approach ...,With a wind-thermal-bundled power system (WTBP...,None,Abdelrahman
...,...,...,...,...,...
289,290,A Dynamic Equivalent Method for PMSG-WTG Based...,The dynamic security assessment of power syste...,None,Pablo
292,293,Applying High-Performance Computing to the Eur...,This work considers the European Resource Adeq...,None,Abdelrahman
294,295,Combined Data-Driven and Model-Driven Location...,This article addresses the low-delay location ...,None,Andrey
295,296,Comprehensive-Contribution-Based Primary Frequ...,The rising penetration of renewable energy lea...,None,Gabriel


In [17]:

batch1_indices = set(selected_abstracts_df["Source_Index"].astype(int))

batch2_candidates = (
    max_affinity.loc[
        max_affinity["PRP_Name"].eq("IBR")
        & ~max_affinity["Abstract_Index"].isin(batch1_indices)
    ]
    .sort_values(["agreement_max", "Final_Affinity", "Abstract_Index"], ascending=[False, False, True])
    .drop_duplicates(subset=["Abstract_Index"])
    .head(15)
    .copy()
)

selected_abstracts_batch_2 = (
    abstracts_df.loc[abstracts_df["Source_Index"].isin(batch2_candidates["Abstract_Index"])]
    [["Source_Index", "Document Title", "Abstract"]]
    .drop_duplicates(subset=["Source_Index"])
    .copy()
)

selected_abstracts_batch_2["Program"] = None
selected_abstracts_batch_2["assigned to"] = [persons[i % len(persons)] for i in range(len(selected_abstracts_batch_2))]
selected_abstracts_batch_2["PRP_Name"] = "IBR"

selected_abstracts_batch_2 = selected_abstracts_batch_2.sort_values(["assigned to", "Source_Index"]).reset_index(drop=True)
selected_abstracts_batch_2.to_csv("selected_abstracts_batch_2.csv", index=False)

display(selected_abstracts_batch_2)

,Source_Index,Document Title,Abstract,Program,assigned to,PRP_Name
0,20,Dual Grid-Forming Control With Energy Regulati...,"To integrate weak onshore grids, some grid for...",None,Abdelrahman,IBR
1,124,Online Inertia Allocation for Grid-Connected R...,Frequency security is one of the challenges fo...,None,Abdelrahman,IBR
2,158,Modeling of Frequency Security Constraints and...,The high penetration of converter-based renewa...,None,Abdelrahman,IBR
3,263,Enhanced DC Voltage and Frequency Regulation f...,This work proposes a primary frequency support...,None,Abdelrahman,IBR
4,86,RoCoF Constrained Unit Commitment Considering ...,The post-disturbance rate of change of frequen...,None,Andrey,IBR
5,145,Interaction Between Grid-Forming Converters Wi...,Grid-forming converters (GFC) are promising te...,None,Andrey,IBR
6,168,Dynamic Stabilization of a Centralized Weak Gr...,This paper addresses the instabilities and dyn...,None,Andrey,IBR
7,269,Real-Time Detection and Localization of Line T...,Line trip events widely exist in power systems...,None,Andrey,IBR
8,114,A Coordinated Voltage-Frequency Support Method...,Voltage and frequency stability is one of the ...,None,Gabriel,IBR
9,150,Transient Stability Analysis and Improvement f...,"As a widely configured controller, the limiter...",None,Gabriel,IBR


In [18]:
ra_adjudication_aggregated_df

,Abstract_Index,PRP_Name,Final_Affinity,RA2025_ALL_ID,agreement_max,RA2025_ID,Affinity_Level
0,1,CROF,58.000000,27,0.666667,27,Moderate
1,1,DER Bulk Services,9.166667,18,1.000000,18,Low
2,1,DER C&C,21.000000,12,1.000000,12,Low
3,1,IBR,10.833333,1,1.000000,1,Low
4,1,Planning,87.333333,41,1.000000,41,High
...,...,...,...,...,...,...,...
2095,300,DER C&C,30.000000,15,1.000000,15,Low
2096,300,IBR,85.000000,4,1.000000,4,High
2097,300,Planning,40.000000,46,1.000000,46,Low
2098,300,Stability,62.000000,9,0.500000,9,Moderate
